# S03 — Stateful Streaming and Delta

**Time: about 75 minutes.**
Covers: Data Engineering with Structured Streaming · Managing Streaming Tables with Delta Lake

### What you will be able to do afterwards

- Aggregate a stream over event-time windows, not arrival time.
- Set a watermark and explain exactly what it throws away.
- Join a stream to a slowly changing dimension.
- Use `foreachBatch` to do something a streaming sink cannot do on its own.
- Read a Delta table as a stream and explain why that resembles a log broker.

### Why this is the hard one

Everything so far was stateless: each row processed independently, nothing remembered. From
here Spark holds state between micro-batches, and state that grows without bound is the
main way streaming jobs die in production. The watermark is how you bound it.

In [ ]:
from pyspark.sql import DataFrame, functions as F
from helpers import utils, event_stream, streaming_utils

src = utils.get_configs("web_events")
events_silver = src["table_silver"]

win = utils.get_configs("events_per_minute")
window_table, window_checkpoint = win["table_silver"], win["checkpoint_path"]

enr = utils.get_configs("events_enriched")
enriched_table, enriched_checkpoint = enr["table_silver"], enr["checkpoint_path"]

fun = utils.get_configs("funnel_by_product")
funnel_table, funnel_checkpoint = fun["table_gold"], fun["checkpoint_path"]

products_silver = f"{src['catalog']}.{src['schema_silver']}.products"
users_silver = f"{src['catalog']}.{src['schema_silver']}.users"

spark = utils.spark
manifest = event_stream.read_manifest()
print(events_silver, window_table, enriched_table, funnel_table, sep="\n")

## Step 0 — A Delta table is a streaming source

**TO DO**

1. Read `events_silver` as a stream: `spark.readStream.table(events_silver)`.
2. Confirm `isStreaming`.
3. Read `DESCRIBE HISTORY` on that table and note the version numbers.

**The idea worth taking away.** A Delta table is an append-ordered log with numbered
versions. Reading it as a stream means starting at a version and moving forward — versions
are offsets. That is structurally what a log broker gives you, which is why so many
Lakehouse pipelines have no broker at all: bronze *is* the queue.

Options worth knowing: `startingVersion`, `startingTimestamp`, `maxBytesPerTrigger`,
`ignoreChanges`.

> **Question:** name one thing a real broker gives you that a Delta table does not. Be
> specific — "it's faster" is not an answer.

In [ ]:
# TO DO: read the silver table as a stream and confirm it is streaming


# TO DO: look at the versions available to stream from

## Step 1 — Windowed aggregation with a watermark

**TO DO**

Build `events_per_minute` in silver: event counts per one-minute tumbling window, per
`event_type`.

1. Stream from `events_silver`.
2. `withWatermark("event_timestamp", "5 minutes")` — before the `groupBy`, not after.
3. `groupBy(F.window("event_timestamp", "1 minute"), "event_type")`, count.
4. Flatten the window struct into `window_start` and `window_end` columns.
5. Write in **append** mode with `window_checkpoint`, `trigger(availableNow=True)`.

**What the watermark actually does.** It tracks the maximum event time seen so far and
subtracts your threshold. Windows entirely older than that are considered final: their
state is dropped, and any event arriving for them afterwards is discarded. Without it,
state grows forever. With it, you have chosen a completeness/memory trade-off.

The seed data stamps about 2% of events roughly 25 minutes in the past specifically so a
5-minute watermark drops something. If your totals match the source exactly, your watermark
is not doing anything.

> **Questions:**
> - Append mode is legal here but was illegal in S01 Step 4. What changed?
> - Set the watermark to 60 minutes and re-run into a different table. How does the total
>   change, and what did you pay for the extra completeness?
> - How would you find out how many events were dropped? (`streaming_utils.state_summary`
>   is a start.)

In [ ]:
def events_per_minute(source_table: str, watermark: str = "5 minutes") -> DataFrame:
    """
    One-minute tumbling counts per event_type, watermarked on event_timestamp.

    Args:
        source_table: silver events table to stream from.
        watermark: how late an event may be and still be counted.
    Returns:
        Streaming DataFrame with window_start, window_end, event_type, event_count.
    """
    # TO DO
    pass


# TO DO: run it, then compare the windowed total to the source row count

## Step 2 — Stream-static join

Events carry `product_id` and `user_id` but no names. Enrich them from the batch tables you
built in the batch track.

**TO DO**

Build `events_enriched` in silver with the event fields plus `product_name`,
`product_category`, `product_price` and `user_name`.

1. Stream from `events_silver`.
2. Read `products_silver` and `users_silver` as **static** DataFrames — plain
   `spark.table()`, no `readStream`.
3. **Left** join on `product_id` and on `user_id`.
4. Write append with `enriched_checkpoint`.

**Left, not inner.** `view` events have a null `product_id` by design — the user looked at a
listing page, not a product. An inner join silently deletes over half your traffic, and the
check will catch it.

**How a stream-static join behaves.** The static side is re-read on every micro-batch, so
dimension updates appear without restarting. There is no state and no watermark, because
the stream side is never held.

> **Questions:**
> - The static side is re-read each batch. What does that cost, and how would you notice it?
> - A product is added to the dimension after an event referencing it has already been
>   processed. Does that event ever get its name? What would you build if it had to?
> - If you needed the price *as it was at event time* rather than the current price, what
>   would you join to instead? (You built exactly that in B04.)

In [ ]:
def enrich_events(source_table: str, products: str, users: str) -> DataFrame:
    """Stream-static left join of events against the product and user dimensions."""
    # TO DO
    pass


# TO DO: run it, then confirm no events were lost

## Step 3 — `foreachBatch` and a MERGE

You want a gold funnel table: per product, how many views, clicks, add-to-carts and
purchases. That is an upsert, and a streaming sink cannot `MERGE` on its own.

`foreachBatch` hands you each micro-batch as an ordinary batch DataFrame. Inside it you can
do anything you can do in batch — merge, write to two tables, call an API.

**TO DO**

1. Create `funnel_by_product` in gold: `product_id`, `product_name`, `views`, `clicks`,
   `add_to_carts`, `purchases`, `updated_at`.
2. Write `upsert_funnel(batch_df, batch_id)` that aggregates the batch by product and
   `MERGE`s into the target, **adding** the batch counts to the existing ones.
3. Stream `events_enriched` through it with `.foreachBatch(upsert_funnel)`.
4. Verify the funnel is monotonic: views ≥ clicks ≥ add_to_carts ≥ purchases per product.

**The trap.** `foreachBatch` gives at-least-once, not exactly-once. If the job fails after
your merge commits but before the checkpoint advances, that batch is replayed and your
`+` accumulation double-counts. `batch_id` is the defence: it is stable across retries, so
you can record which batch ids you have already applied and skip them.

> **Questions:**
> - Implement the `batch_id` guard, or explain precisely how you would. Where does the
>   record of applied batches live?
> - Your merge accumulates with `+`. What would change if it replaced instead, and which is
>   correct here?

In [ ]:
def upsert_funnel(batch_df: DataFrame, batch_id: int) -> None:
    """
    Aggregate one micro-batch by product and merge the counts into the gold funnel.

    Args:
        batch_df: an ordinary batch DataFrame — one micro-batch of the stream.
        batch_id: monotonically increasing, stable across retries of the same batch.
    """
    # TO DO
    pass


# TO DO: create the funnel table, run the foreachBatch stream, verify monotonicity

## Step 4 — Watch the state

**TO DO**

1. Re-run the Step 1 windowed query with a `processingTime` trigger over a few batches.
2. Call `streaming_utils.state_summary(query)` and put it in a DataFrame.
3. Look at `rows_total`, `rows_updated` and `rows_removed` per batch.
4. Now run the same query with the watermark removed and compare `rows_total`.

> **Questions:**
> - With a watermark, `rows_removed` is non-zero and `rows_total` stabilises. Without it,
>   what shape does `rows_total` take, and what happens to this job after a week?
> - `memoryUsedBytes` is in the same output. At what value would you start worrying, and
>   what would you change first?

In [ ]:
# TO DO: run with processingTime, capture state_summary, compare with and without watermark

## Before you finish

In [ ]:
streaming_utils.stop_all_streams()

## Checks

In [ ]:
from helpers import test_runner

test_runner.run("S03-stateful-streaming")

## Recap

- Windows are on event time. Arrival time is a property of your infrastructure, not of the
  business event.
- A watermark is a deliberate trade of completeness for bounded memory. Unbounded state is
  how streaming jobs die.
- Stream-static joins re-read the static side each batch, need no watermark, and should
  almost always be left joins.
- `foreachBatch` unlocks batch operations inside a stream, and hands you the at-least-once
  problem in exchange. `batch_id` is the tool for it.
- Reading Delta as a stream makes bronze the queue. Versions are offsets.